# 12 Download river levels (Rijkswaterstaat)

<!-- navigation -->
[<kbd>← 11 Download BRO groundwater levels</kbd>](11%20Download%20BRO%20groundwater%20levels.ipynb) &nbsp; [<kbd>Overview</kbd>](README.md) &nbsp; [<kbd>15 Overview of measurements →</kbd>](15%20Overview%20of%20measurements.ipynb)

**Optional**: instead of, or in addition to, notebook 10

Downloads the observed river levels from Rijkswaterstaat (Matroos) into the pastastore of each transect.

| | |
|---|---|
| **Input** | river gauge per transect in the transects metadata (`loc_river`) |
| **Output** | pastastore per transect |

Downloads the observed water levels of the river gauges from Rijkswaterstaat (Matroos), with [hydropandas](https://hydropandas.readthedocs.io) (`hpd.WaterlvlObs.from_matroos`), and stores them in the [pastastore](https://pastastore.readthedocs.io) of each transect that uses them. The series get the name of the gauge as in the metadata. Times are converted to Dutch winter time (UTC+1). Matroos interprets the requested period in UTC, so a series may start one hour later than requested.

In [ ]:
from pathlib import Path

import timflow_levee_toolkit as tlt

import hydropandas as hpd
import matplotlib.pyplot as plt
import pandas as pd
tlt.show_versions()

In [ ]:
## UPDATE FOR YOUR PROJECT: the project folder (with project_settings.toml and the data)
project_dir = Path.cwd().parent / "examples" / "papendrecht"

project = tlt.load_project(project_dir)
units = project.units
project

In [ ]:
## UPDATE FOR YOUR PROJECT: transects and download period
metadata = project.metadata()
transects = project.transects()  # or a selection, e.g. ["Papendrecht West"]

# Download period: None = all available data. The model period is chosen later, in notebook 51,
# from the figures of the complete series. The example downloads only one week, to keep it small.
tmin = pd.Timestamp("2025-12-01")  # None: from the first measurement
tmax = pd.Timestamp("2025-12-08")  # None: up to the last measurement
print(f"Transects: {transects}, download period: {tmin} - {tmax}")

## 1. Download the river gauges
The gauge names are Matroos location names, e.g. `dordrecht`. To find names: `hydropandas.io.matroos.select_parameters(units="waterlevel", sources="observed", astype="geodataframe")`.

In [ ]:
selected = metadata[metadata["transect"].isin(transects)]
locations = sorted(selected["loc_river"].dropna().unique())  # transects can share a gauge

river_obs = {}
for location in locations:
    try:
        river_obs[location] = hpd.WaterlvlObs.from_matroos(
            location=location, unit="waterlevel", source="observed", tmin=tmin, tmax=tmax
        )
        print(f"{location}: {river_obs[location].iloc[:, 0].count()} values")
    except Exception as e:
        print(f"{location}: not retrieved ({type(e).__name__}: {e})")

In [ ]:
f, ax = plt.subplots(figsize=(10, 4))
for location, o in river_obs.items():
    ax.plot(o.index, o.iloc[:, 0], label=location)
ax.set_title("River levels (RWS)", loc="right")
ax.set_xlabel("Date")
ax.set_ylabel(f"water level [{units['head']}]")
ax.legend(loc=(0, 1), frameon=False, ncol=4, fontsize="x-small")
ax.grid(True)

## 2. Store in the pastastore of each transect

In [ ]:
overwrite = False  # True: replace series that are already in the pastastore

for _, row in selected.iterrows():
    gauge = row["loc_river"]
    if gauge not in river_obs:
        print(f"{row['transect']}: no river levels for '{gauge}'")
        continue
    obs = river_obs[gauge]
    series = obs.iloc[:, 0].rename(gauge).to_frame()
    series_metadata = pd.DataFrame(
        {"x": [obs.x], "y": [obs.y], "kind": "river", "source": "Rijkswaterstaat (Matroos)"}, index=[gauge]
    )
    pstore = project.open_pastastore(row["transect"])
    added = tlt.import_timeseries(pstore, series, metadata=series_metadata, overwrite=overwrite)
    print(f"{row['transect']}: {added or 'nothing'} added")

---

<!-- navigation -->
[<kbd>← 11 Download BRO groundwater levels</kbd>](11%20Download%20BRO%20groundwater%20levels.ipynb) &nbsp; [<kbd>Overview</kbd>](README.md) &nbsp; [<kbd>15 Overview of measurements →</kbd>](15%20Overview%20of%20measurements.ipynb)